# bsc_03 — Headroom của NGỮ CẢNH BỀ MẶT (trước khi xây graph)

Stage 1 thất bại với cơ chế: **ảo giác ở vùng `absent` chiếm ~82% khối lượng lỗi vùng mỏng**;
recall 81% (sụn dày) → 37% (sụn mỏng). Xem `STAGE1_CONCLUSION.md`.

**Giả thuyết sửa:** cho mỗi node thấy **láng giềng trên bề mặt xương** (plan §4.4 Step 5 —
intra-surface message passing) để biết *"cả vùng này không có sụn"*. Tia 1D vứt bỏ ngữ cảnh
tiếp tuyến; lưới trên bề mặt trả lại ngữ cảnh ở thang **cm** — đúng thang mà vùng mất sụn
tồn tại.

**Nhưng plan cảnh báo ngược lại** (§4.4 Step 5 + test MM3): làm mượt trên bề mặt có thể
**lấp luôn focal defect thật**. Với failure mode là ảo giác, nó đi hai chiều ngược nhau:

| | |
|---|---|
| FP **rời rạc** | hàng xóm bỏ phiếu **dập được** → graph hữu ích |
| FP thành **mảng** | hàng xóm **củng cố** cái sai → graph làm tệ hơn |

⇒ **Đo trước, xây sau** — đúng tinh thần M0 (ROI cascade mất một chu kỳ vì xây trước khi đo).

## Bốn phép đo

1. **Coherence** — vắng sụn có liên tục theo không gian không? *(có tín hiệu để khai thác?)*
2. **Neighbor-oracle** — nếu **biết presence GT của láng giềng**, đoán được presence của node
   không? ⇒ **trần** của ngữ cảnh bề mặt.
3. **FP isolation** — FP của model rời rạc hay thành mảng? ⇒ graph giúp hay hại.
4. **Smoothing sweep** — "graph nhà nghèo": làm mượt presence rồi đo lại lỗi vùng mỏng.
   **Không train gì**, dùng checkpoint đã có.

## Cổng quyết định — ghi TRƯỚC khi chạy

| Tiêu chí | Ngưỡng |
|---|---|
| `signal_exists` — coherence lift | > 1.5 |
| `fp_isolated` — % láng giềng đoán đúng quanh FP | > 0.60 |
| `smoothing_helps` — giảm lỗi vùng mỏng | > 0.10mm |

**Cả ba đạt ⇒ CÓ HEADROOM**, đáng xây neighbor-pooling/graph.
**Không đủ ⇒ ngữ cảnh bề mặt không cứu được failure mode này** — thêm một bằng chứng cho
kết luận âm tính, và tiết kiệm cả Stage 2.

### 0. Config

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL, REPO_DIR = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git", "/content/repo"
import os, sys, glob, json
if not os.path.isdir(f"{REPO_DIR}/bsc"):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!pip install -q nibabel SimpleITK 2>/dev/null

# nap lai bsc sau git pull -> khoi restart kernel
for _m in [k for k in list(sys.modules) if k == "bsc" or k.startswith("bsc.")]:
    del sys.modules[_m]

import numpy as np
from tqdm.auto import tqdm
from bsc import core, metrics, headroom, io_utils, model as M, mvp
from bsc import atlas as atlas_mod, experiment as X, surface_context as SC
from bsc.core import RayConfig

BSC_ROOT = "/content/drive/MyDrive/bsc"
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"
cfg = RayConfig()
CART = {"femoral_cart": 2, "med_tib_cart": 4}
BONE = {"femoral_cart": 1, "med_tib_cart": 3}
CLS = "femoral_cart"

SP = tuple(float(x) for x in io_utils.load_nii(
    sorted(glob.glob(f"{RAW}/labelsTr/oaizib_*.nii.gz"))[0])[1])
print("spacing:", tuple(round(x, 4) for x in SP), "| SC san sang:", hasattr(SC, "surface_context_case"))

### 1. Nạp split + atlas + **checkpoint đã khóa**

Dùng lại canonical P2 (`4e133ba5fa0a74c6`) — **không train gì mới**. Nếu chưa có checkpoint
(session mới, chưa chạy Phase A) thì cell tự train lại P2 config cũ.

In [ ]:
SPLITS = f"{BSC_ROOT}/splits/splits_zib_v1_fixed.json"
N_TRAIN, N_VAL = 40, 10

cases_all = sorted(os.path.basename(p)[:-len("_0000.nii.gz")]
                   for p in glob.glob(f"{RAW}/imagesTr/*_0000.nii.gz"))
cases_all = [c for c in cases_all if c.startswith("oaizib_")]
fold_of = json.load(open(SPLITS))["fold_of"]
zib = [c for c in cases_all if c in fold_of]
train_ids = [c for c in zib if fold_of[c] != 0][:N_TRAIN]
val_ids   = [c for c in zib if fold_of[c] == 0][:N_VAL]

CV_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
def baseline_path(cid):
    p = glob.glob(f"{CV_DIR}/fold_*/validation/{cid}.nii.gz")
    return p[0] if p else None

SRC = mvp.NiftiCaseSource(RAW, CLS, CART[CLS], BONE[CLS], baseline_path)
z = np.load(f"{BSC_ROOT}/atlas/atlas_{CLS}_fold0.npz", allow_pickle=True)
ATLAS = atlas_mod.ArticularAtlas(z["prob"], int(z["n_bins"]), float(z["lo"]),
                                 float(z["hi"]), tuple(z["case_ids"]), 0, None)
atlas_mod.assert_no_leak(ATLAS, val_ids)
print(f"train {len(train_ids)} | val {len(val_ids)} | atlas {len(ATLAS.case_ids)} ca")

run = X.from_plan("P2", CLS, seed=1)
ckpt = sorted(glob.glob(f"{BSC_ROOT}/runs/{run.experiment_id}/*/model.pt"))
if ckpt:
    run_dir = os.path.dirname(ckpt[-1])
    net = mvp.load_run_model(run_dir, device="cuda")
    man = mvp.run_manifest(run_dir)
    print(f"Nap checkpoint {man['checkpoint_sha256']} | git {man['git_commit'][:8]}"
          f" | val occ-Dice {man['val_occ_dice']:.3f}")
else:
    print("Chua co checkpoint - train lai P2 config cu (~20 phut)")
    res = mvp.train_run(run, SRC, train_ids, val_ids, cfg, atlas=ATLAS,
                        rays_per_case=20000, epochs=30, device="cuda")
    run_dir = mvp.save_run(res, BSC_ROOT, cfg, epochs=30, lr=3e-4, rays_per_case=20000,
                           dataset_revision="Dataset001_KneeOA", git_cwd=REPO_DIR)
    net = mvp.load_run_model(run_dir, device="cuda")
    print("Da luu ->", run_dir)

### 2. Chạy 4 phép đo trên 10 ca val

Có checkpoint resume — đứt kết nối chạy lại chỉ làm ca còn thiếu.

In [ ]:
K_NB = 8
ALPHAS = (0.0, 0.3, 0.5, 0.7, 1.0)
B0_MM = 0.5517                      # ResEnc baseline, loi bien vung mong
SC_CKPT = f"{run_dir}/surface_context_k{K_NB}.jsonl"

done = ({json.loads(l)["case"] for l in open(SC_CKPT)}
        if os.path.exists(SC_CKPT) else set())
rows = [json.loads(l) for l in open(SC_CKPT)] if done else []
print(f"Da co {len(done)} ca | con {len([c for c in val_ids if c not in done])}")

with open(SC_CKPT, "a") as fh:
    for cid in tqdm(val_ids, desc="surface context"):
        if cid in done:
            continue
        r = SC.surface_context_case(run, net, SRC, cid, cfg, ATLAS, k=K_NB,
                                    alphas=ALPHAS, device="cuda")
        if r:
            rows.append(r)
            fh.write(json.dumps(r, default=str) + chr(10)); fh.flush()

print(f"Xong {len(rows)} ca -> {SC_CKPT}")

### 3. Kết quả + cổng quyết định

In [ ]:
s = SC.summarize_surface_context(rows, baseline_thin_mm=B0_MM)

print(f"n = {s['n']} ca val")
print()
print("[1] COHERENCE - vang sun co lien tuc theo khong gian?")
print(f"    node absent co {s['nb_absent_given_absent']:.1%} lang gieng cung absent"
      f"  (ty le nen {s['base_absent_rate']:.1%})")
print(f"    lift = {s['coherence_lift']:.2f}x   (cong >1.5)   -> {s['signal_exists']}")
print()
print("[2] NEIGHBOR-ORACLE - biet presence GT cua lang gieng thi doan duoc gi?")
print(f"    F1 {s['nb_oracle_f1']:.3f} | absent-recall {s['nb_oracle_absent_recall']:.3f}")
print("    Cao => ngu canh be mat DU de xac dinh presence (tran cao cho graph)")
print()
print("[3] FP ISOLATION - FP roi rac hay thanh mang?")
print(f"    {s['fp_nb_correct']:.1%} lang gieng quanh FP duoc doan DUNG   (cong >60%)"
      f"   -> {s['fp_isolated']}")
print("    >60% => FP roi rac, hang xom DAP duoc | <40% => FP thanh MANG, cung co cai sai")
print()
print("[4] SMOOTHING SWEEP - lam muot presence tren be mat (graph nha ngheo)")
print(f"    {'alpha':>7}{'thin err':>11}{'vs B0':>9}{'presF1':>9}{'absentRec':>11}")
for a, v in s["sweep"].items():
    e = v["thin_err_mm"]
    mark = "  <- tot nhat" if a == s["best_alpha"] else ("  (xoa sach du doan)"
           if a in s["alphas_wiped_out"] else "")
    print(f"    {a:>7}{e:>10.3f}mm{e-B0_MM:>+9.3f}{v['presence_f1']:>9.3f}"
          f"{v['absent_recall']:>11.3f}{mark}")
print(f"    giam duoc {s['smoothing_gain_mm']:+.3f}mm  (cong >0.10)  -> {s['smoothing_helps']}")
print()
print("=" * 70)
print("=> " + s["verdict"])
print("=" * 70)

json.dump(s, open(f"{run_dir}/surface_context_summary.json", "w"), indent=2, default=str)
print(f"Da ghi {run_dir}/surface_context_summary.json")

### 4. Đọc kết quả

**CÓ HEADROOM** (cả 3 cổng đạt) → xây neighbor-pooling: mỗi node gộp embedding của K láng
giềng trước khi vào presence head. Bản rẻ của §4.4 Step 5, chưa cần graph transformer.

**KHÔNG ĐỦ HEADROOM** → ngữ cảnh bề mặt không cứu được ảo giác absent. Ghép với các bằng
chứng đã có (loss/sampling ❌, dữ liệu 3.5× ❌), kết luận âm tính Stage 1 vững thêm — và
tiết kiệm được toàn bộ Stage 2.

**Trường hợp hỗn hợp** (ví dụ tín hiệu có nhưng FP thành mảng) là kết quả *có thông tin*:
nó nói ảo giác của model **có cấu trúc không gian**, tức model sai một cách nhất quán theo
vùng — gợi ý vấn đề nằm ở **tín hiệu ảnh tại vùng đó**, không phải ở thiếu ngữ cảnh.

⚠️ Mọi số ở đây đo trên **10 ca val** đã dùng nhiều lần cho chẩn đoán ⇒ là **engineering
diagnostic**, không phải final Gate. Nếu quyết định xây, phải đánh giá lại trên split phân
tầng (Phase C).